<a href="https://colab.research.google.com/github/Burakcelebi2003/xwrc-plus-calculator/blob/main/xwRC%2B_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install streamlit numpy scipy pandas plotly

In [ ]:
#xwRC+ Hessaplama
%%writefile sabermetrics_gui.py
import streamlit as st

st.set_page_config(
    page_title="Postseason xwRC+ Hesaplayıcı",
    page_icon="⚾",
    layout="centered"
)

st.title("⚾xwRC+ Hesaplayıcı")
st.caption("Ham wOBA → Postseason Scale → Tahmini wRC+")


st.header("1.Hücum Verileri")

st.info(
    "Olay ağırlıkları ölçekleme öncesi ağırlıklar olmalıdır. "
    "Standart wOBA ağırlıklarını doğrudan kullanırsan "
    "hesaplanan Scale geçerli olmaz."
)

with st.form("raw_woba_form"):

    st.subheader("Hücum olayları")

    col1, col2 = st.columns(2)

    with col1:
        pa = st.number_input(
            "PA",
            min_value=1,
            value=100,
            step=1
        )

        bb = st.number_input(
            "BB",
            min_value=0,
            value=10,
            step=1
        )

        ibb = st.number_input(
            "IBB",
            min_value=0,
            value=1,
            step=1
        )

        hbp = st.number_input(
            "HBP",
            min_value=0,
            value=2,
            step=1
        )

    with col2:
        single = st.number_input(
            "1B",
            min_value=0,
            value=15,
            step=1
        )

        double = st.number_input(
            "2B",
            min_value=0,
            value=5,
            step=1
        )

        triple = st.number_input(
            "3B",
            min_value=0,
            value=1,
            step=1
        )

        hr = st.number_input(
            "HR",
            min_value=0,
            value=4,
            step=1
        )

    st.subheader("Ölçekleme öncesi olay ağırlıkları")

    w_bb = st.number_input(
        "BB ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    w_hbp = st.number_input(
        "HBP ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    w_1b = st.number_input(
        "1B ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    w_2b = st.number_input(
        "2B ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    w_3b = st.number_input(
        "3B ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    w_hr = st.number_input(
        "HR ağırlığı",
        min_value=0.0,
        value=0.0,
        format="%.5f"
    )

    calculate_raw = st.form_submit_button(
        "Ham wOBA'yı hesapla"
    )

if calculate_raw:

    event_counts = [
        bb, hbp, single, double, triple, hr
    ]

    weights = [
        w_bb, w_hbp, w_1b, w_2b, w_3b, w_hr
    ]

    if ibb > bb:
        st.error("IBB, BB değerinden büyük olamaz.")

    elif pa <= ibb:
        st.error("PA, IBB değerinden büyük olmalıdır.")

    elif sum(event_counts) > pa:
        st.error(
            "Girdiğin olayların toplamı PA değerini aşıyor. "
            "Verileri kontrol et."
        )

    elif any(weight <= 0 for weight in weights):
        st.error(
            "Altı olay ağırlığının tamamı pozitif olmalıdır."
        )

    else:
        denominator = pa - ibb

        numerator = (
            w_bb * bb
            + w_hbp * hbp
            + w_1b * single
            + w_2b * double
            + w_3b * triple
            + w_hr * hr
        )

        raw_woba = numerator / denominator

        st.session_state["raw_woba"] = raw_woba

        st.success("Ham wOBA hesaplandı.")

        st.metric(
            "Ham wOBA",
            f"{raw_woba:.5f}"
        )

        st.latex(
            r"\text{Ham wOBA}"
            r"=\frac{\sum(w_i \cdot N_i)}{PA-IBB}"
        )

        st.code(
            f"{numerator:.5f} / {denominator}"
            f" = {raw_woba:.5f}"
        )

        st.warning(
            "Bu, PA − IBB paydasını kullanan yaklaşık "
            "bir hesaplamadır. Standart wOBA paydasında "
            "SH ve catcher interference gibi durumlar "
            "da dikkate alınabilir."
        )


# ==================================================
# 2. POSTSEASON wOBA SCALE
# ==================================================

st.divider()
st.header("2.wOBA Scale")

if "raw_woba" not in st.session_state:

    st.warning(
        "Önce ilk bölümde ham wOBA hesaplamalısın."
    )

else:

    raw_woba = st.session_state["raw_woba"]

    st.write(
        f"Hesaplanan ham wOBA: **{raw_woba:.5f}**"
    )

    postseason_obp = st.number_input(
        "Postseason OBP",
        min_value=0.001,
        max_value=1.0,
        value=0.293,
        step=0.001,
        format="%.3f",
        key="postseason_obp"
    )

    if st.button("Postseason Scale hesapla"):

        if raw_woba <= 0:
            st.error(
                "Ham wOBA sıfırdan büyük olmalıdır."
            )

        else:
            scale = postseason_obp / raw_woba

            st.session_state["postseason_scale"] = scale

            st.latex(
                r"\text{Scale}_{PS}"
                r"=\frac{\text{Postseason OBP}}"
                r"{\text{Ham wOBA}_{PS}}"
            )

            st.code(
                f"{postseason_obp:.5f} / "
                f"{raw_woba:.5f} = {scale:.5f}"
            )

            st.metric(
                "Postseason wOBA Scale",
                f"{scale:.5f}"
            )

            st.warning(
                "Bu katsayı, girilen ağırlıklar gerçekten "
                "ölçekleme öncesi ağırlıklarsa anlamlıdır. "
                "OBP düzeltmesi ve kullanılan payda nedeniyle "
                "sonuç yaklaşık bir postseason katsayısıdır."
            )


# ==================================================
# 3. TAHMİNİ wRC+
# ==================================================

st.divider()
st.header("3.xwOBA → Tahmini wRC+")

if "postseason_scale" not in st.session_state:

    st.warning(
        "Önce ikinci bölümde Postseason Scale hesapla."
    )

else:

    scale = st.session_state["postseason_scale"]

    st.write(
        f"Kullanılan Postseason Scale: **{scale:.5f}**"
    )

    xwoba = st.number_input(
        "Oyuncunun xwOBA değeri",
        min_value=0.0,
        max_value=1.0,
        value=0.389,
        step=0.001,
        format="%.3f"
    )

    league_woba = st.number_input(
        "Referans lig wOBA",
        min_value=0.0,
        max_value=1.0,
        value=0.282,
        step=0.001,
        format="%.3f"
    )

    r_pa = st.number_input(
        "Lig R/PA",
        min_value=0.001,
        max_value=2.0,
        value=0.096,
        step=0.001,
        format="%.3f"
    )

    st.subheader("Hesaplama formülü")

    formula = (
        r"\widehat{wRC^+}=100+"
        r"100\frac{xwOBA-\text{Lg wOBA}}"
        r"{\text{Scale}\times\text{Lg R/PA}}"
    )

    st.latex(formula)

    st.caption(
        "Aşağıdaki işlem, xwOBA'yı wOBA benzeri bir "
        "ölçekte kullanan yaklaşık bir indekstir."
    )

    st.code(
        f"100 + 100 * ({xwoba:.3f} - "
        f"{league_woba:.3f}) / "
        f"({scale:.5f} * {r_pa:.3f})"
    )

    if st.button("Tahmini indeksi hesapla"):

        estimate = (
            100
            + 100 * (xwoba - league_woba)
            / (scale * r_pa)
        )

        st.metric(
            "Tahmini xwOBA tabanlı indeks",
            f"{estimate:.1f}"
        )

        st.write(
            f"xwOBA: **{xwoba:.3f}**  \n"
            f"Referans lig wOBA: **{league_woba:.3f}**  \n"
            f"Postseason Scale: **{scale:.5f}**  \n"
            f"Lig R/PA: **{r_pa:.3f}**"
        )

        st.warning(
            "Bu sonuç resmî wRC+ veya doğrulanmış xwRC+ "
            "değildir. Gerçek wRC+ hesabında wOBA tabanlı "
            "run üretimi, lig ve park düzeltmeleri kullanılır. "
            "xwOBA'dan doğrudan dönüşüm ayrıca kalibre edilmelidir."
        )

In [ ]:
import urllib
print("Şifre:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

In [ ]:
import subprocess
import os
import urllib.request
import platform
import re
import time
import threading

system = platform.system()
binary_name = "cloudflared.exe" if system == "Windows" else "cloudflared"

if not os.path.exists(binary_name):
    print("Cloudflare tünel aracı indiriliyor")
    if system == "Windows":
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
    else:
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, binary_name)
    if system != "Windows":
        os.chmod(binary_name, 0o755)
    print("İndirme tamamlandı!")

def run_streamlit():
    subprocess.run(["streamlit", "run", "sabermetrics_gui.py", "--server.headless=true"])

streamlit_thread = threading.Thread(target=run_streamlit, daemon=True)
streamlit_thread.start()
time.sleep(3)

print("Cloudflare tüneli açılıyor")
cmd = [os.path.abspath(binary_name), "tunnel", "--url", "http://localhost:8501"]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

tunnel_url = None
for _ in range(15):
    line = process.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n==================================================")
    print(f"BAĞLANTI HAZIR!")
    print(f"{tunnel_url}")
    print(f"==================================================\n")
else:
    print("Tünel linki otomatik yakalanamadı")